## 数据采集（MNIST）

- 这一节负责：下载/读取 MNIST，并把像素归一化到 `[0, 1]`。
- 写报告时：这里对应 **数据采集** 模块。


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    """Load MNIST and normalize images to [0, 1].

    Returns:
        (x, y), (x_test, y_test)
        x/x_test: float32, shape (N, 28, 28)
        y/y_test: int64, shape (N,)
    """
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    # normalize to [0, 1] and cast to float32 for TF training
    x = x.astype(np.float32) / 255.0
    x_test = x_test.astype(np.float32) / 255.0
    
    return (x, y), (x_test, y_test)

In [ ]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

## 模型描述（两层全连接网络 FNN）

- 输入：`(N, 28, 28)` 灰度图，先 flatten 为 `784` 维向量。
- 结构：`784 -> hidden(ReLU) -> 10(logits)`。
- 写报告时：这里对应 **模型描述** 模块。


In [ ]:
class myModel:
    def __init__(self):
        """Two-layer fully-connected network for MNIST.

        - 输入 x: shape (N, 28, 28), dtype float32, 值域 [0, 1]
        - 输出 logits: shape (N, 10), 未经过 softmax 的分类分数

        这里不用 Keras Layer/Model，仅用 tf.Variable + 手写前向，
        便于你在报告中说明参数与计算图。
        """

        # 参数维度约定：
        # W1: (784, hidden_dim)  b1: (hidden_dim,)
        # W2: (hidden_dim, 10)   b2: (10,)
        hidden_dim = 100

        # 小方差随机初始化可以让训练更稳定
        self.W1 = tf.Variable(tf.random.normal([28 * 28, hidden_dim], stddev=0.1), name='W1')
        self.b1 = tf.Variable(tf.zeros([hidden_dim]), name='b1')
        self.W2 = tf.Variable(tf.random.normal([hidden_dim, 10], stddev=0.1), name='W2')
        self.b2 = tf.Variable(tf.zeros([10]), name='b2')
    def __call__(self, x):
        """前向计算，返回未归一化 logits。

        x: (N, 28, 28)
        return: logits (N, 10)
        """
        x = tf.cast(x, tf.float32)
        x = tf.reshape(x, [-1, 28 * 28])

        # hidden layer
        h1 = tf.matmul(x, self.W1) + self.b1
        h1 = tf.nn.relu(h1)

        # output layer (logits)
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

# 这里选择 Adam 作为优化器（也可以换 SGD/Momentum）
optimizer = optimizers.Adam(learning_rate=1e-3)

## 函数定义：损失与评估指标

- 写报告时：这里对应 **函数定义** 模块（loss/accuracy 等）。


In [ ]:
@tf.function
def compute_loss(logits, labels):
    # logits: (N, 10), labels: (N,) 取值 0~9
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    # 预测类别 = logits 最大值所在下标
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    # 使用优化器更新参数（对应你报告里的“训练方法/优化器”描述）
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 拟合效果（训练/测试指标输出）

- 这一节会打印每个 epoch 的 loss/accuracy，以及最终 test 集的指标。
- 写报告时：这里对应 **拟合效果** 模块。


In [ ]:
import matplotlib.pyplot as plt

train_data, test_data = mnist_dataset()

# 记录每个 epoch 的损失和准确率
history_loss = []
history_acc = []

# 训练循环：每个 epoch 在整个训练集上做一次参数更新（这里用的是 full-batch）
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    history_loss.append(loss.numpy())
    history_acc.append(accuracy.numpy())
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
    
test_loss, test_accuracy = test(model, 
                                tf.constant(test_data[0], dtype=tf.float32), 
                                tf.constant(test_data[1], dtype=tf.int64))

print('test loss', test_loss.numpy(), '; accuracy', test_accuracy.numpy())

# 可视化损失和准确率
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 绘制 Loss 曲线
axes[0].plot(range(50), history_loss, label='Train Loss', color='blue')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# 绘制 Accuracy 曲线
axes[1].plot(range(50), history_acc, label='Train Accuracy', color='orange')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()